# SRC-001 Cross-Source Consistency Audit

## 1. Setup

**Objective:** Evaluate the consistency between network PCAP captures and lower-layer TXT telemetry in the SRC-001 dataset.

**Scope:**
- Verify file pairing across selected scenarios.
- Examine basic structural consistency.
- Assess temporal alignment feasibility.
- Identify limitations affecting Root Cause Analysis.

**Out of Scope:**
- Packet-level synchronization.
- Comprehensive traffic analysis.
- Root cause identification.

**Method:** Use Python and Pandas to inspect the original files without modifying them.

In [ ]:
import pandas as pd
import json
import subprocess

from src.data.inventory import DataInventory
from src.data.pcap_tools import PacketTools

SOURCE_ID = "SRC-001"

inventory = DataInventory(source_id=SOURCE_ID)
RAW_DATA_PATH = inventory.raw_data_path
packet_tools = PacketTools("capinfos", "tshark").get_packet_tools()

print(RAW_DATA_PATH)

## 2. Source Pairing

**Objective:** Determine whether each selected PCAP scenario has a corresponding TXT telemetry file.

**Method:** Discover PCAP and TXT files dynamically and compare their scenario identifiers using file names.

**Limitation:** Matching file names indicate a potential scenario-level relationship, but do not establish temporal synchronization or content consistency.

In [2]:
pcap_files = sorted(
    path for path in RAW_DATA_PATH.iterdir()
    if path.is_file() and path.suffix.lower() == ".pcap"
)

txt_files = {
    path.stem: path
    for path in RAW_DATA_PATH.iterdir()
    if path.is_file() and path.suffix.lower() == ".txt"
}

pair_inventory = pd.DataFrame([
    {
        "scenario": path.stem,
        "pcap_exists": True,
        "txt_exists": path.stem in txt_files,
        "txt_file": txt_files[path.stem].name if path.stem in txt_files else None
    }
    for path in pcap_files
])

display(pair_inventory)

,scenario,pcap_exists,txt_exists,txt_file
0,benign,True,True,benign.txt
1,ddos_icmp_hping3,True,True,ddos_icmp_hping3.txt
2,ftp_hydra,True,True,ftp_hydra.txt
3,icmp_nping,True,True,icmp_nping.txt
4,portscan_masscan,True,True,portscan_masscan.txt
5,web_xss,True,True,web_xss.txt


### Finding — Source Pairing

All six selected scenarios have matching PCAP and TXT files based on their scenario identifiers.

No missing source pairs were identified.

**Result: PASS — 6/6 matched scenarios.**

File-name correspondence confirms scenario-level availability but does not establish temporal synchronization or consistency between packet captures and telemetry records.

## 3. Structural Consistency

**Objective:** Verify that each matched scenario contains non-empty PCAP and TXT files.

**Method:** Compare file availability and sizes without loading their contents into memory.

**Scope:** File-level consistency only. Detailed format validation was addressed in the previous audits.

In [7]:
assert pair_inventory["txt_exists"].all(), "Missing TXT pairs"
pair_summary = pd.DataFrame([
    {
        "scenario": path.stem,
        "pcap_mb": round(path.stat().st_size / (1024 ** 2), 2),
        "txt_mb": round(txt_files[path.stem].stat().st_size / (1024 ** 2), 2),
        "valid_pair": path.stat().st_size > 0 and txt_files[path.stem].stat().st_size > 0
    }
    for path in pcap_files if path.stem in txt_files
])

display(pair_summary)
print(f"Non-empty pairs: {pair_summary['valid_pair'].sum()}/{len(pcap_files)}")

,scenario,pcap_mb,txt_mb,valid_pair
0,benign,67.81,0.81,True
1,ddos_icmp_hping3,213.82,0.51,True
2,ftp_hydra,384.83,0.21,True
3,icmp_nping,481.71,0.36,True
4,portscan_masscan,833.01,0.39,True
5,web_xss,134.01,0.08,True


Non-empty pairs: 6/6


### Finding — Structural Consistency

All six selected scenarios contain non-empty PCAP and TXT files.

PCAP file sizes range from 67.81 MB to 833.01 MB, while TXT telemetry files range from 0.08 MB to 0.81 MB.

**Result: PASS — 6/6 non-empty source pairs.**

Both sources are available for subsequent analysis. However, file availability and size alone do not establish temporal alignment or semantic consistency between network traffic and telemetry.

## 4. Temporal Consistency

**Objective:** Determine whether PCAP captures and TXT telemetry can be temporally aligned.

**Method:** Inspect available timestamp fields and compare their temporal coverage.

**Scope:** Scenario-level temporal consistency only. Packet-to-telemetry synchronization is excluded.

In [4]:
for scenario, path in sorted(txt_files.items()):
    print(f"\nScenario: {scenario}")
    with path.open("r", encoding="utf-8", errors="replace") as file:
        for _, line in zip(range(3), file):
            print(line.strip()[:200])


Scenario: benign
{"ue_id": 2, "timestamp": 1751374264245, "cellid": 11111111, "in_sync": 1, "rnti": "097a", "dlBytes": 541944552, "dlMcs": 23, "dlBler": 0.064554, "ulBytes": 821528571, "ulMcs": 11, "ulBler": 0.161862,
{"ue_id": 15, "timestamp": 1751374264245, "cellid": 11111111, "in_sync": 1, "rnti": "de81", "dlBytes": 14016622, "dlMcs": 9, "dlBler": 0.0, "ulBytes": 69276604, "ulMcs": 9, "ulBler": 0.070186, "ri": 1
{"ue_id": 2, "timestamp": 1751374265245, "cellid": 11111111, "in_sync": 1, "rnti": "097a", "dlBytes": 541974006, "dlMcs": 24, "dlBler": 0.178451, "ulBytes": 821535047, "ulMcs": 11, "ulBler": 0.069676,

Scenario: ddos_icmp_hping3
{"ue_id": 1, "timestamp": 1751364899238, "cellid": 11111111, "in_sync": 1, "rnti": "c66b", "dlBytes": 36435132, "dlMcs": 6, "dlBler": 0.687809, "ulBytes": 57489253, "ulMcs": 6, "ulBler": 1.0, "ri": 1,
{"ue_id": 4, "timestamp": 1751364899238, "cellid": 11111111, "in_sync": 1, "rnti": "2fe4", "dlBytes": 231572599, "dlMcs": 16, "dlBler": 0.075167, "ulB

In [5]:
rows = []

for pcap in pcap_files:
    txt = txt_files[pcap.stem]

    result = subprocess.run(
        [packet_tools["capinfos"], "-T", "-r", "-S", "-a", "-e", str(pcap)],
        capture_output=True, text=True, check=True
    )

    fields = result.stdout.strip().split("\t")
    pcap_start, pcap_end = map(float, fields[-2:])

    with txt.open(encoding="utf-8") as file:
        timestamps = [
            int(json.loads(line)["timestamp"])
            for line in file if line.strip()
        ]

    txt_start = min(timestamps) / 1000
    txt_end = max(timestamps) / 1000

    overlap = max(0, min(pcap_end, txt_end) - max(pcap_start, txt_start))

    rows.append({
        "scenario": pcap.stem,
        "pcap_start": pd.to_datetime(pcap_start, unit="s", utc=True),
        "pcap_end": pd.to_datetime(pcap_end, unit="s", utc=True),
        "txt_start": pd.to_datetime(txt_start, unit="s", utc=True),
        "txt_end": pd.to_datetime(txt_end, unit="s", utc=True),
        "overlap_seconds": round(overlap, 2)
    })

temporal_summary = pd.DataFrame(rows)
display(temporal_summary)

,scenario,pcap_start,pcap_end,txt_start,txt_end,overlap_seconds
0,benign,2025-07-01 11:56:59.243052006+00:00,2025-07-01 11:58:35.625381947+00:00,2025-07-01 12:51:04.244999886+00:00,2025-07-01 13:09:17.835000038+00:00,0.00
1,ddos_icmp_hping3,2025-07-01 12:26:34.829547882+00:00,2025-07-01 12:30:53.316622972+00:00,2025-07-01 10:14:59.237999916+00:00,2025-07-01 10:23:18.308000088+00:00,0.00
2,ftp_hydra,2025-06-27 13:12:25.695770979+00:00,2025-06-27 13:26:22.073431015+00:00,2025-06-27 13:16:18.796999931+00:00,2025-06-27 13:26:18.796999931+00:00,600.00
3,icmp_nping,2025-06-17 13:23:05.073112965+00:00,2025-06-17 13:33:11.446136951+00:00,2025-06-27 14:53:59.759000063+00:00,2025-06-27 15:11:15.759000063+00:00,0.00
4,portscan_masscan,2025-06-25 17:26:18.523513079+00:00,2025-06-25 17:36:28.581932068+00:00,2025-06-25 17:26:09.844000101+00:00,2025-06-25 17:44:55.844000101+00:00,610.06
5,web_xss,2025-06-30 11:16:37.477566004+00:00,2025-06-30 11:20:41.317579985+00:00,2025-06-30 11:16:33.280999899+00:00,2025-06-30 11:20:37.280999899+00:00,239.80


In [6]:
temporal_summary["temporal_overlap"] = temporal_summary["overlap_seconds"] > 0

display(temporal_summary[
    ["scenario", "overlap_seconds", "temporal_overlap"]
])

print(f"Overlapping pairs: {temporal_summary['temporal_overlap'].sum()}/{len(temporal_summary)}")

,scenario,overlap_seconds,temporal_overlap
0,benign,0.00,False
1,ddos_icmp_hping3,0.00,False
2,ftp_hydra,600.00,True
3,icmp_nping,0.00,False
4,portscan_masscan,610.06,True
5,web_xss,239.80,True


Overlapping pairs: 3/6


## 5. Audit Findings

### 5.1. Source Availability

All six selected scenarios contain corresponding PCAP and TXT files. No missing or empty source pairs were identified.

**Result: PASS — 6/6 source pairs.**

### 5.2. Temporal Consistency

Temporal analysis identified overlapping PCAP and TXT intervals in three scenarios:

- `ftp_hydra`: 600.00 seconds.
- `portscan_masscan`: 610.06 seconds.
- `web_xss`: 239.80 seconds.

The remaining scenarios (`benign`, `ddos_icmp_hping3`, and `icmp_nping`) exhibit no temporal overlap.

**Result: PARTIAL — 3/6 overlapping pairs.**

Temporal overlap indicates potential alignment but does not establish event-level synchronization.

### 5.3. Identified Limitations

- File-name correspondence does not guarantee semantic consistency.
- Three scenarios lack overlapping capture and telemetry timestamps.
- Clock synchronization and capture conditions have not been independently verified.
- Packet-to-telemetry relationships cannot be established from the available metadata alone.

### 5.4. Conclusion and Dataset Suitability

**Decision: Conditionally Suitable**

The selected PCAP and TXT files are suitable for scenario-level analysis within the DAICP MVP.

However, direct temporal correlation between both sources is supported by overlapping timestamps in only three of the six scenarios, and synchronization remains unverified.

Scenarios without temporal overlap should not be treated as synchronized evidence without additional validation.

The dataset remains a candidate for the MVP, subject to evaluation-risk assessment and the selection of scenarios with sufficient supporting evidence.

**Next step:** Evaluate data leakage risks and establish constraints for constructing the evaluation dataset.